# 第五次作业：sklearn 感知机源码解析（`Perceptron`）

> 主题：阅读 scikit-learn 的实现。

> **主线**：只讲 `linear_model.Perceptron`；PassiveAggressive 与 hinge-SVM 仅作对照，不展开。

环境：`scikit-learn 1.3.2` / Python 3.8。文中行号均以 **1.3.2** 版本的库为准。

**sklearn 的 `Perceptron` 类没有单独实现训练循环，而是复用 `BaseSGDClassifier` 的 SGD 框架：固定部分内部参数、设置默认值、缩减公开参数，从而提供感知机接口。**


## 1. 感知机

对每个样本 $(x_i, y_i)$，$y_i \in \{-1,+1\}$。记**决策值** $f(x)=w^\top x + b$，**带符号间隔** $z = y f(x)$；感知机在 $z \le 0$ 时更新

$$w \leftarrow w + \eta\, y_i x_i,\qquad b \leftarrow b + \eta\, y_i$$

理论上，感知机由误分类驱动、无正则、不加权、没有概率输出、只有在线性可分时才保证收敛。

`sklearn/linear_model/_perceptron.py` **整个文件只有 221 行**；其中 `Perceptron` **类体占 213 行**（L9–L221），类体里 docstring 又占 158 行——**扣除类文档字符串后还剩 55 行，其中包含空行和注释**（这 55 行不含文件顶部的 8 行 import）。它主要做的是裁剪公开参数并委托基类。我们先看整体调用关系，再逐条拆开。


## 2. 一次 `fit` 的调用关系

`Perceptron` 中没有定义 `fit`，调用的是基类的方法。函数调用关系如下：

```text
fit → _fit → _partial_fit
                 ├─ 二分类：_fit_binary → fit_binary
                 └─ 多分类：_fit_multiclass → 并行调用 fit_binary
                                              ↓
                                      _plain_sgd32/64
                                              ↓
                                         Hinge.dloss
```

对应源码说明（`_stochastic_gradient.py`，行号由 `inspect.getsourcelines` 取得）：

- `BaseSGDClassifier.fit`：L877，被 `@_fit_context` 装饰；
- `_fit`：L655，重置 `t_`、处理 `warm_start`，再交给 `_partial_fit`；
- `_partial_fit`：L571，计算 `class_weight`，按类别数分流；
- `_fit_binary`：L723（二分类）；`_fit_multiclass`：L757（多分类，joblib 并行）；
- 模块级 `fit_binary`：L359，准备数据集、解析随机种子；
- `_get_plain_sgd_function`：L494，float64 取 `_plain_sgd64`、float32 取 `_plain_sgd32`。

三点概括：

1. **公开参数被缩减**：`loss` 固定为 `"perceptron"`，其余训练逻辑都在基类；
2. **两条分叉**：二分类走 `_fit_binary`，多分类走 `_fit_multiclass`，后者并行调用模块级的 `fit_binary`；
3. **训练循环在编译模块里**：`_sgd_fast` 是 `.so`，只有 `Hinge` 这个损失对象可以用 `py_loss` / `py_dloss` 从 Python 侧读到判据。

`fit` 上的 `@_fit_context` 装饰器，导致「越界参数要到 `fit` 期才报错」。


In [1]:
import inspect
import re

import numpy as np
from sklearn.linear_model import Perceptron
from sklearn.linear_model._stochastic_gradient import (
    BaseSGDClassifier,
    _get_plain_sgd_function,
    fit_binary,
)


def line_of(name, source, start):
    """找出 `def name` 的行号"""
    for i, l in enumerate(source):
        if re.match(r"\s*def %s\b" % name, l):
            return start + i
    return -1


cls_src, cls_start = inspect.getsourcelines(BaseSGDClassifier)
fname = inspect.getsourcefile(BaseSGDClassifier).split("/")[-1]

print("== 方法归属 ==")
print("  'fit' 由 Perceptron 自己定义 :", "fit" in Perceptron.__dict__)
print("  Perceptron.fit 实际指向      :", Perceptron.fit.__qualname__)
print("  Perceptron 自己定义的成员    :", [k for k in Perceptron.__dict__ if not k.startswith("__")])
print()
print("== 调用链（源码文件 %s）==" % fname)
for name in ("fit", "_fit", "_partial_fit", "_fit_binary", "_fit_multiclass"):
    print("  BaseSGDClassifier.%-17s L%d" % (name, line_of(name, cls_src, cls_start)))
print("  %-35s L%d" % ("fit_binary", inspect.getsourcelines(fit_binary)[1]))
print("  %-35s L%d" % ("_get_plain_sgd_function", inspect.getsourcelines(_get_plain_sgd_function)[1]))
print()
print("  float64:", _get_plain_sgd_function(np.dtype("float64")).__name__)
print("  float32:", _get_plain_sgd_function(np.dtype("float32")).__name__)
print("  训练循环:", _get_plain_sgd_function(np.dtype("float64")).__module__)


== 方法归属 ==
  'fit' 由 Perceptron 自己定义 : False
  Perceptron.fit 实际指向      : BaseSGDClassifier.fit
  Perceptron 自己定义的成员    : ['_parameter_constraints', 'set_fit_request', 'set_partial_fit_request', 'set_score_request', '_abc_impl']

== 调用链（源码文件 _stochastic_gradient.py）==
  BaseSGDClassifier.fit               L877
  BaseSGDClassifier._fit              L655
  BaseSGDClassifier._partial_fit      L571
  BaseSGDClassifier._fit_binary       L723
  BaseSGDClassifier._fit_multiclass   L757
  fit_binary                          L359
  _get_plain_sgd_function             L494

  float64: _plain_sgd64
  float32: _plain_sgd32
  训练循环: sklearn.linear_model._sgd_fast


## 3. 定位源码：文件 221 行，类体 213 行

`Perceptron` 的全部实现都在 `sklearn/linear_model/_perceptron.py`，其中不新增训练逻辑：调整参数约束表、设置默认值，然后调用 `BaseSGDClassifier`。

**文件 221 行**，其中**类体 213 行**（L9–L221）；类体里 docstring 158 行，**扣除类文档字符串后还剩 55 行，其中包含空行和注释**。

`linear_model/` 目录下的 `.py` 都可 `inspect.getsource`；`.so` 只有二进制；`.pxd` 只是给 `_sag_fast` 用的声明，**不含 Hinge**。


In [2]:
import ast
import inspect
import os
import sys

import sklearn
from sklearn.linear_model import Perceptron

sk_dir = os.path.dirname(sklearn.__file__)
lm_dir = os.path.join(sk_dir, "linear_model")
path = inspect.getsourcefile(Perceptron)
text = open(path, encoding="utf-8").read()

print("python       :", sys.version.split()[0])
print("sklearn      :", sklearn.__version__)
print("perceptron   :", path)
print()

tree = ast.parse(text)
cls = next(n for n in tree.body if isinstance(n, ast.ClassDef) and n.name == "Perceptron")
doc = cls.body[0]
total = cls.end_lineno - cls.lineno + 1
doc_lines = doc.end_lineno - doc.lineno + 1

print("== 行数：文件 vs 类体（两把尺子）==")
print("  文件总行数       : %d" % len(text.splitlines()))
print("  类体             : %d 行（L%d–L%d）" % (total, cls.lineno, cls.end_lineno))
print("  类体内 docstring : %d 行（L%d–L%d）" % (doc_lines, doc.lineno, doc.end_lineno))
print("  类体其余实现     : %d 行（不含文件顶部 import）" % (total - doc_lines))
print("  文件顶部 import  : %d 行" % (cls.lineno - 1))
print()

print("== linear_model 目录 ==")
for suffix, label in ((".py", "纯 Python"), (".so", "编译产物"), (".pxd", "Cython 声明")):
    files = sorted(f for f in os.listdir(lm_dir) if f.endswith(suffix))
    print("%-10s (%d): %s" % (label, len(files), ", ".join(files) if files else "-"))


python       : 3.8.20
sklearn      : 1.3.2
perceptron   : /root/MSA/ENTER/envs/vmunet/lib/python3.8/site-packages/sklearn/linear_model/_perceptron.py

== 行数：文件 vs 类体（两把尺子）==
  文件总行数       : 221
  类体             : 213 行（L9–L221）
  类体内 docstring : 158 行（L10–L167）
  类体其余实现     : 55 行（不含文件顶部 import）
  文件顶部 import  : 8 行

== linear_model 目录 ==
纯 Python   (17): __init__.py, _base.py, _bayes.py, _coordinate_descent.py, _huber.py, _least_angle.py, _linear_loss.py, _logistic.py, _omp.py, _passive_aggressive.py, _perceptron.py, _quantile.py, _ransac.py, _ridge.py, _sag.py, _stochastic_gradient.py, _theil_sen.py
编译产物       (3): _cd_fast.cpython-38-x86_64-linux-gnu.so, _sag_fast.cpython-38-x86_64-linux-gnu.so, _sgd_fast.cpython-38-x86_64-linux-gnu.so
Cython 声明  (1): _sgd_fast.pxd


## 4. 公开参数被缩减：签名里没有 `loss` 与 `average`

`Perceptron` 复用基类的训练实现，通过**固定部分内部参数、设置默认值和缩减公开参数**，提供感知机接口。其中，`loss` 和 `learning_rate` 固定，`eta0` 默认值为 1.0，但允许用户修改。

- 「用户不能改损失、不能开权重平均（ASGD）」的**直接原因**是**构造函数签名里根本没有 `loss` 和 `average` 这两个形参**，传入即构造期 `TypeError`；
- `Perceptron.__init__` 把 `loss="perceptron"`、`learning_rate="constant"` 固定下来，并给 `eta0=1.0`、`power_t=0.5` 设置默认值（`alpha`、`penalty`、`shuffle`、`tol` 等仍可由用户配置）；
- 类体里的 `_parameter_constraints.pop("loss")` / `.pop("average")` **只是移除这两个键对应的参数校验约束**（它们继承自 `BaseSGDClassifier`，签名里既然没有，就不该再校验）。

这样 `Perceptron()` 与 `SGDClassifier(loss="perceptron", eta0=1.0, learning_rate="constant", penalty=None)` 在数值近似意义下一致（`np.allclose`，不是逐位相等）。

**参数校验的时机：**`loss` / `average` 根本不在签名里，是**构造期** `TypeError`；而 `eta0=0` / `alpha=-1` 这类越界值是**构造通过、`fit` 期才报错**（sklearn 1.3 的参数校验挂在 `fit` 的装饰器上）。先看源码，再实测。


In [3]:
import inspect

from sklearn.linear_model import Perceptron

MARKS = {
    'self._parameter_constraints.pop("loss")': "  # <== 只删掉 loss 的校验约束",
    'self._parameter_constraints.pop("average")': "  # <== 只删掉 average 的校验约束",
    'loss="perceptron",': "  # <== 固定损失",
    'learning_rate="constant",': "  # <== 固定学习率表",
    'eta0=1.0,': "  # <== 默认步长 1.0（允许修改）",
    'power_t=0.5,': "  # <== constant 下不生效，但仍固定",
}

src = inspect.getsource(Perceptron).splitlines()
start = next(i for i, l in enumerate(src) if l.strip().startswith("_parameter_constraints"))
end = next(i for i, l in enumerate(src) if l.strip().startswith("def __init__"))
print("== 参数约束表（类体片段）==")
for line in src[start:end]:
    print(line + MARKS.get(line.strip(), ""))

print()
print("== __init__ ==")
for line in inspect.getsource(Perceptron.__init__).splitlines():
    print(line + MARKS.get(line.strip(), ""))

sig = inspect.signature(Perceptron.__init__)
print()
print("== 签名证据：限制来自签名，而不是 pop() ==")
print("  'loss' in Perceptron.__init__ 签名   :", "loss" in sig.parameters)
print("  'average' in Perceptron.__init__ 签名:", "average" in sig.parameters)

c = Perceptron._parameter_constraints
print()
print("== 结果 ==")
print("  'loss' in constraints   :", "loss" in c)
print("  'average' in constraints:", "average" in c)
def describe(cons):
    """把 sklearn 的约束对象翻译成可读文本（避免打印出 <... object at 0x...>）。"""
    out = []
    for con in cons:
        if con is None:
            out.append("None")
        elif hasattr(con, "options"):          # StrOptions
            out.append("{" + ", ".join(sorted(map(repr, con.options))) + "}")
        elif hasattr(con, "left"):             # Interval
            lo = "-inf" if con.left is None else con.left
            hi = "inf" if con.right is None else con.right
            left_br = "[" if con.closed in ("left", "both") else "("
            right_br = "]" if con.closed in ("right", "both") else ")"
            out.append("%s%s, %s%s" % (left_br, lo, hi, right_br))
        else:
            out.append(repr(con))
    return " 或者 ".join(out)


print("  penalty 允许值          :", describe(c["penalty"]))
print("  eta0 允许范围           :", describe(c["eta0"]))
print("  alpha 允许范围          :", describe(c["alpha"]))


== 参数约束表（类体片段）==
    _parameter_constraints: dict = {**BaseSGDClassifier._parameter_constraints}
    _parameter_constraints.pop("loss")
    _parameter_constraints.pop("average")
    _parameter_constraints.update(
        {
            "penalty": [StrOptions({"l2", "l1", "elasticnet"}), None],
            "alpha": [Interval(Real, 0, None, closed="left")],
            "l1_ratio": [Interval(Real, 0, 1, closed="both")],
            "eta0": [Interval(Real, 0, None, closed="left")],
        }
    )


== __init__ ==
    def __init__(
        self,
        *,
        penalty=None,
        alpha=0.0001,
        l1_ratio=0.15,
        fit_intercept=True,
        max_iter=1000,
        tol=1e-3,
        shuffle=True,
        verbose=0,
        eta0=1.0,  # <== 默认步长 1.0（允许修改）
        n_jobs=None,
        random_state=0,
        early_stopping=False,
        validation_fraction=0.1,
        n_iter_no_change=5,
        class_weight=None,
        warm_start=False,
    ):
        super().__init__(
   

In [4]:
import warnings

import numpy as np
from sklearn.datasets import make_blobs
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Perceptron

X, y = make_blobs(n_samples=80, centers=2, random_state=0)

CASES = [
    ("average=True", dict(average=True)),
    ("loss='hinge'", dict(loss="hinge")),
    ("eta0=0.0", dict(eta0=0.0)),
    ("alpha=-1.0", dict(alpha=-1.0)),
    ("penalty='l1'", dict(penalty="l1")),
]

print("== 五种参数写法的报错时机 ==")
for label, kw in CASES:
    try:
        est = Perceptron(**kw)
    except Exception as e:
        print("%-14s -> 构造期 %-20s %s" % (label, type(e).__name__, str(e).splitlines()[0][:70]))
        continue
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            est.fit(X, y)
        print("%-14s -> 构造通过，fit 成功（coef[:2]=%s）" % (label, np.round(est.coef_[0][:2], 3)))
    except Exception as e:
        print("%-14s -> 构造通过，fit 期 %-18s %s" % (label, type(e).__name__, str(e).splitlines()[0][:70]))

print()
base = Perceptron(random_state=0).fit(X, y)
l1 = Perceptron(penalty="l1", random_state=0, max_iter=200).fit(X, y)
print("默认 penalty=None 的 coef[:2]:", np.round(base.coef_[0][:2], 3))
print("penalty='l1'      的 coef[:2]:", np.round(l1.coef_[0][:2], 3), "-> 已带正则")

== 五种参数写法的报错时机 ==
average=True   -> 构造期 TypeError            __init__() got an unexpected keyword argument 'average'
loss='hinge'   -> 构造期 TypeError            __init__() got an unexpected keyword argument 'loss'
eta0=0.0       -> 构造通过，fit 期 ValueError         eta0 must be > 0
alpha=-1.0     -> 构造通过，fit 期 InvalidParameterError The 'alpha' parameter of Perceptron must be a float in the range [0.0,
penalty='l1'   -> 构造通过，fit 成功（coef[:2]=[ 3.851 -8.56 ]）

默认 penalty=None 的 coef[:2]: [ 3.955 -8.664]
penalty='l1'      的 coef[:2]: [ 3.851 -8.56 ] -> 已带正则


## 5. 「感知机损失」
### 5.1 `Hinge()`

`loss_functions` 表里两行紧挨着：

```python
"hinge":      (Hinge, 1.0),
"perceptron": (Hinge, 0.0),
```

同一个 Cython 类，只差一个 `double` 阈值。沿用前面的记号：决策值 $f(x)=w^\top x + b$，带符号间隔 $z = y f(x)$。

- `Hinge(0)`：`z <= 0` 就更新，这就是感知机的判据；
- `Hinge(1)`：`z <= 1` 就更新，对应线性 SVM 常用的 hinge 损失；完整训练行为还取决于正则化和学习率设置。

实测 `Hinge(1).py_dloss(1.0, 1.0) == -1.0`，即 `z = 1` 也更新——两个折点都取等号。

进一步看：`loss` / `dloss` 是 `cdef` 方法，Python 侧没有这两个名字——编译模块只额外暴露了专供测试的 `py_loss` / `py_dloss` 包装，正好可以**直接读出**判据，不必靠行为反推。


In [5]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.linear_model import Perceptron
from sklearn.linear_model._stochastic_gradient import BaseSGDClassifier

print("== loss_functions 表 ==")
for name, spec in BaseSGDClassifier.loss_functions.items():
    cls, args = spec[0], spec[1:]
    print("  %-28s -> %s%s" % (name, cls.__name__, args if args else ""))

X, y = make_blobs(n_samples=40, centers=2, random_state=0)
lf = Perceptron().fit(X, y).loss_function_
print()
print("Perceptron().loss_function_ :", type(lf).__module__ + "." + type(lf).__name__)
print("Python 侧有 .loss 吗？      :", hasattr(lf, "loss"))
print("Python 侧公开属性           :", [a for a in dir(lf) if not a.startswith("__")])
print("py_loss 的 self 说明        :", lf.py_loss.__doc__.splitlines()[0])


== loss_functions 表 ==
  hinge                        -> Hinge(1.0,)
  squared_hinge                -> SquaredHinge(1.0,)
  perceptron                   -> Hinge(0.0,)
  log_loss                     -> Log
  modified_huber               -> ModifiedHuber
  squared_error                -> SquaredLoss
  huber                        -> Huber(0.1,)
  epsilon_insensitive          -> EpsilonInsensitive(0.1,)
  squared_epsilon_insensitive  -> SquaredEpsilonInsensitive(0.1,)

Perceptron().loss_function_ : sklearn.linear_model._sgd_fast.Hinge
Python 侧有 .loss 吗？      : False
Python 侧公开属性           : ['py_dloss', 'py_loss']
py_loss 的 self 说明        : Python version of `loss` for testing.


In [6]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.linear_model import Perceptron, SGDClassifier

X, y = make_blobs(n_samples=40, centers=2, random_state=0)
lf_p = Perceptron().fit(X, y).loss_function_                  # Hinge(0.0)
lf_h = SGDClassifier(loss="hinge").fit(X, y).loss_function_    # Hinge(1.0)

ROWS = [(1.5, 1.0), (1.0, 1.0), (0.5, 1.0), (0.0, 1.0), (-0.5, 1.0), (-1.5, 1.0), (0.5, -1.0)]

print("== 直接读编译模块里的判据（py_loss / py_dloss）==")
print("  p 是 w^T x + intercept，z = p * y；两个折点 z=0 与 z=1 都列进来")
print()
print("%6s %3s | %10s %11s | %10s %11s" % ("p", "y", "P.py_loss", "P.py_dloss", "H.py_loss", "H.py_dloss"))
for p, yy in ROWS:
    print("%6.2f %3.0f | %10.3f %11.3f | %10.3f %11.3f"
          % (p, yy, lf_p.py_loss(p, yy), lf_p.py_dloss(p, yy),
             lf_h.py_loss(p, yy), lf_h.py_dloss(p, yy)))

print()
print("Perceptron：dloss 在 z <= 0 时非零 -> 分界恒为 0（z = 0 也更新）")
print("hinge     ：dloss 在 z <= 1 时非零 -> 分界在间隔 1（z = 1 也更新）")
print()
print("两个折点都取等号，且损失值为 0 而次梯度非零：")
print("  p=0.00, y=+1 -> Hinge(0): py_loss=%.1f, py_dloss=%+.1f"
      % (lf_p.py_loss(0.0, 1.0), lf_p.py_dloss(0.0, 1.0)))
print("  p=1.00, y=+1 -> Hinge(1): py_loss=%.1f, py_dloss=%+.1f"
      % (lf_h.py_loss(1.0, 1.0), lf_h.py_dloss(1.0, 1.0)))
print("  p=1.00 再大一点 y=+1 -> Hinge(1): py_dloss=%+.1f（越过折点，立刻变 0）"
      % lf_h.py_dloss(float(np.nextafter(1.0, 2.0)), 1.0))


== 直接读编译模块里的判据（py_loss / py_dloss）==
  p 是 w^T x + intercept，z = p * y；两个折点 z=0 与 z=1 都列进来

     p   y |  P.py_loss  P.py_dloss |  H.py_loss  H.py_dloss
  1.50   1 |      0.000       0.000 |      0.000       0.000
  1.00   1 |      0.000       0.000 |      0.000      -1.000
  0.50   1 |      0.000       0.000 |      0.500      -1.000
  0.00   1 |      0.000      -1.000 |      1.000      -1.000
 -0.50   1 |      0.500      -1.000 |      1.500      -1.000
 -1.50   1 |      1.500      -1.000 |      2.500      -1.000
  0.50  -1 |      0.500       1.000 |      1.500       1.000

Perceptron：dloss 在 z <= 0 时非零 -> 分界恒为 0（z = 0 也更新）
hinge     ：dloss 在 z <= 1 时非零 -> 分界在间隔 1（z = 1 也更新）

两个折点都取等号，且损失值为 0 而次梯度非零：
  p=0.00, y=+1 -> Hinge(0): py_loss=0.0, py_dloss=-1.0
  p=1.00, y=+1 -> Hinge(1): py_loss=0.0, py_dloss=-1.0
  p=1.00 再大一点 y=+1 -> Hinge(1): py_dloss=+0.0（越过折点，立刻变 0）


In [7]:
import numpy as np
from sklearn.linear_model import Perceptron, SGDClassifier


def probe(margin, estimator):
    X = np.array([[1.0], [5.0]])
    y = np.array([1, 0])
    sw = np.array([1.0, 0.0])
    est = estimator.fit(
        X, y,
        sample_weight=sw,
        coef_init=np.array([0.0]),
        intercept_init=np.array([float(margin)]),
    )
    return est.coef_.ravel().copy(), est.intercept_.copy()


print("== 同一初始决策值下，是否更新？==")
print("%8s | %-30s | %-30s" % ("margin", "Perceptron (theta=0)", "hinge (theta=1)"))
for m in (0.5, 2.0, -0.5):
    cp, bp = probe(m, Perceptron(shuffle=False, max_iter=1, tol=None))
    ch, bh = probe(m, SGDClassifier(loss="hinge", penalty=None, learning_rate="constant",
                                    eta0=1.0, shuffle=False, max_iter=1, tol=None))
    print("%8.2f | coef=%-8s b=%5.2f %-4s | coef=%-8s b=%5.2f %-4s"
          % (m, np.round(cp, 2), bp[0], "更新" if not np.allclose(cp, 0) else "不动",
             np.round(ch, 2), bh[0], "更新" if not np.allclose(ch, 0) else "不动"))

print()
print("== 边界：决策值恰好为 0 ==")
X0 = np.array([[0.0], [0.0]])
y0 = np.array([1, 0])
sw0 = np.array([1.0, 0.0])
p0 = Perceptron(shuffle=False, max_iter=1, tol=None).fit(
    X0, y0, sample_weight=sw0,
    coef_init=np.array([0.0]), intercept_init=np.array([0.0]))
print("  Perceptron intercept_ =", p0.intercept_, "-> 0 分界也会更新（<= 0）")

== 同一初始决策值下，是否更新？==
  margin | Perceptron (theta=0)           | hinge (theta=1)               
    0.50 | coef=[0.]     b= 0.50 不动   | coef=[1.]     b= 1.50 更新  
    2.00 | coef=[0.]     b= 2.00 不动   | coef=[0.]     b= 2.00 不动  
   -0.50 | coef=[1.]     b= 0.50 更新   | coef=[1.]     b= 0.50 更新  

== 边界：决策值恰好为 0 ==
  Perceptron intercept_ = [1.] -> 0 分界也会更新（<= 0）


### 5.2 零初始化时损失已经是 0，为什么还会更新？

训练一开始 `w = 0`、`b = 0`，所有样本在初始模型下的决策值均为零，带符号间隔 `z = 0`。查上面的表：`Hinge(0)` 在 `z = 0` 处 `py_loss = 0`，但 `py_dloss = -y`（即 `∓1`）。

原因是 `max(0, -z)` 在 `z = 0` 处**不可导**（折点），实现必须挑一个次梯度，而 sklearn 挑的是**非零的那一侧**。因此零初始化时，训练处理第一个样本就会采用非零次梯度进行更新；此后是否更新，取决于当前参数下的带符号间隔——一次更新立刻改变了后续样本面对的 `w`。

这也正是更新规则写成 `<= 0`（而不是 `< 0`）的原因：零点不可微，实现选择了非零次梯度。

反过来看也成立：如果把判据改成 `< 0`，零初始化下第一个样本就不会命中，循环会因没有触发更新而提前结束，但这不代表分类正确。


In [8]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.linear_model import Perceptron

X0 = np.array([[1.0], [2.0], [3.0], [4.0]])
y0 = np.array([1, 1, -1, -1])
lf = Perceptron(tol=None).fit(X0, y0).loss_function_

print("== w=0, b=0 时每个样本的 z 都是 0 ==")
for xi, yi in zip(X0.ravel(), y0):
    print("  x=%.1f y=%+d -> z=0 : py_loss=%.1f, py_dloss=%+.1f"
          % (xi, yi, lf.py_loss(0.0, float(yi)), lf.py_dloss(0.0, float(yi))))

z = Perceptron(shuffle=False, max_iter=1, tol=None, fit_intercept=False).fit(X0, y0)
print()
print("  一轮（shuffle=False）后 coef_ =", np.round(z.coef_[0], 3))
print("  手工推演（同一判据 z <= 0）：")
w = 0.0
for xi, yi in zip(X0.ravel(), y0):
    zz = yi * xi * w
    hit = zz <= 0
    if hit:
        w += yi * xi
    print("    x=%.1f y=%+d  z=%+.1f  %s  w=%.1f"
          % (xi, yi, zz, "更新" if hit else "跳过", w))


== w=0, b=0 时每个样本的 z 都是 0 ==
  x=1.0 y=+1 -> z=0 : py_loss=0.0, py_dloss=-1.0
  x=2.0 y=+1 -> z=0 : py_loss=0.0, py_dloss=-1.0
  x=3.0 y=-1 -> z=0 : py_loss=0.0, py_dloss=+1.0
  x=4.0 y=-1 -> z=0 : py_loss=0.0, py_dloss=+1.0

  一轮（shuffle=False）后 coef_ = [-2.]
  手工推演（同一判据 z <= 0）：
    x=1.0 y=+1  z=+0.0  更新  w=1.0
    x=2.0 y=+1  z=+2.0  跳过  w=1.0
    x=3.0 y=-1  z=-3.0  更新  w=-2.0
    x=4.0 y=-1  z=+8.0  跳过  w=-2.0


## 6. 对偶形式：代码里没有，但可以事后重建

感知机对偶形式，是用「每个样本被更新了几次」代替权重向量：

$$w=\sum_i \alpha_i y_i x_i,\qquad \alpha_i \ge 0$$

判错时只需 `α_i += 1`，判定时算 $\sum_j \alpha_j y_j (x_j \cdot x_i)$，于是内积可以换成核函数。**sklearn 的实现中却没有使用这个简化公式**。

### 6.1 代码中没有对偶的痕迹

- `Perceptron.__init__` 的签名里**没有** `kernel`、`dual` 两个形参（`SGDClassifier` 也没有）；
- fit 之后的属性只有 `coef_ / intercept_ / n_iter_ / t_ / loss_function_ / classes_ / n_features_in_`，**没有** `dual_coef_`、`support_`、`n_support_`；
- 作为对照，`SVC(kernel="linear")` 才有 `dual_coef_`、`n_support_`——那是 libsvm 在**对偶问题**上做 SMO 的结果。

### 6.2 复杂度：对偶在「线性 + 大规模」下并不省计算

| | 每次更新 | 每次判定 | 额外内存 |
|---|---|---|---|
| 原始（维护 `w`） | `O(d)` | `O(d)` | `O(d)` |
| 对偶（维护 `α`，缓存 Gram） | `O(1)` | `O(n)` | `O(n²)` |

对偶把成本从「写入」挪到了「读取」。感知机的定位是线性、大规模、在线（`n ≫ d`），这时 `O(n)` 的判定比 `O(d)` 贵一到两个数量级，还要额外 `O(n²)` 内存。只有 `n ≪ d` **并且需要核函数**时，对偶才开始划算。

### 6.3 其他理由

1. **核化收益用不上**：`Perceptron` 是线性分类器，`decision_function` 直接算 `X @ coef_.T + intercept`，没有把内积抛到外面计算的机制。需要核时 sklearn 让人改用 `SVC(kernel=...)`，或用 `Nystroem` / `RBFSampler` 做核近似再套 SGD。
2. **与整个 SGD 家族共享一条内核**：`Perceptron` 只是 `BaseSGDClassifier` 的一个预设档位，和 `SGDClassifier`、`PassiveAggressive`、`Ridge` 共用 `_plain_sgd32/64`。这个内核维护显式的 `w`，损失只是 `loss_functions` 表里的一行。为单个 loss 另写一条对偶路径，等于为了 1/8 的情况复制一套无法表达 `penalty` / `learning_rate` / `average` / 稀疏分支的状态机。
3. **`α` 的「整数计数」语义会被后续开关破坏**：
   - 开 `penalty` 后每步 `w` 都要收缩，`w` 不再是样本的整数倍线性组合；
   - `learning_rate` 变化时，系数与全局步数 `t` 耦合，`α` 不再是纯计数；
   - `average=True`（ASGD）要对 `w` 做时间平均，用 `α` 表达会变成对更新历史的二次累加；
   - `partial_fit` 要增量扩展 Gram 矩阵，而对偶正是在线学习最不擅长的场景。

而 `Perceptron` 恰好只固定了 `loss` 与 `learning_rate`，`penalty`、`average`、`shuffle`、`tol` 全部开放——**它公开允许配置的东西，正是 `α` 表示消化不了的部分**。

### 6.4 但恒等式客观成立

`eta0=1.0` 且 `penalty=None` 时，每次更新就是 `w += y_i x_i`，于是

$$w=\sum_i c_i y_i x_i,\qquad c_i=\text{样本 } i \text{ 触发更新的次数}$$

也就是说对偶解**一直存在**，只是从未作为训练状态存下来。下面的实验把这个恒等式重建出来，并验证它在改步长、加正则后如何失效。


In [9]:
import inspect

import numpy as np
from sklearn.datasets import make_blobs
from sklearn.linear_model import Perceptron
from sklearn.svm import SVC


def manual_counts(X, y, eta=1.0, alpha=0.0, max_epochs=200):
    """教科书循环，额外记录 c_i（隐含的对偶系数），并可选施加 L2 收缩。"""
    c = np.zeros(len(y))
    w = np.zeros(X.shape[1])
    b = 0.0
    for _ in range(max_epochs):
        mistakes = 0
        for i in range(len(y)):
            if alpha:
                w *= (1.0 - eta * alpha)          # L2 的收缩步骤
            if y[i] * (X[i] @ w + b) <= 0:
                w += eta * y[i] * X[i]
                b += eta * y[i]
                c[i] += 1
                mistakes += 1
        if mistakes == 0:
            break
    return w, b, c


X, y = make_blobs(n_samples=60, centers=2, random_state=0)
y = np.where(y == 0, -1, 1)
n, d = X.shape

print("== 规模对照：原始 vs 对偶 ==")
print("  n=%d, d=%d" % (n, d))
print("  原始：每次判定 %d 次乘加，状态 %d 个数" % (d, d))
print("  对偶：每次判定 %d 次乘加，状态 %d 个数，Gram 矩阵 %d×%d" % (n, n, n, n))
print()

print("== 证据：签名与属性里没有对偶 ==")
sig = inspect.signature(Perceptron.__init__)
print("  签名有 kernel / dual ?", "kernel" in sig.parameters, "/", "dual" in sig.parameters)
sk = Perceptron(shuffle=False, tol=None, max_iter=200, random_state=0).fit(X, y)
print("  fit 后属性         :", sorted(a for a in vars(sk) if a.endswith("_")))
print("  有 dual_coef_ / support_ ?", hasattr(sk, "dual_coef_"), "/", hasattr(sk, "support_"))
sv = SVC(kernel="linear").fit(X, y)
print("  SVC(kernel=linear)  : dual_coef_%s, n_support_%s" % (sv.dual_coef_.shape, sv.n_support_))
print()

print("== 恒等式重建：eta=1、无正则 ==")
w_m, b_m, c = manual_counts(X, y)
w_dual = (X * (c * y)[:, None]).sum(axis=0)
b_dual = c[y == 1].sum() - c[y == -1].sum()
print("  sklearn coef_               :", np.round(sk.coef_[0], 4))
print("  Σ c_i·y_i·x_i               :", np.round(w_dual, 4))
print("  最大偏差                    :", float(np.abs(sk.coef_[0] - w_dual).max()))
print("  allclose                    :", bool(np.allclose(sk.coef_[0], w_dual)))
print("  intercept_                  :", sk.intercept_[0], "| 由 c_i 重建:", b_dual)
print("  被用到的样本数 (c_i>0)      : %d / %d" % (int((c > 0).sum()), n))
print("  -> 对偶系数是整数计数，但 sklearn 只存 w")
print()

print("== eta≠1：系数变成 eta × c_i ==")
sk_half = Perceptron(eta0=0.5, shuffle=False, tol=None, max_iter=200, random_state=0).fit(X, y)
w_hm, b_hm, c_half = manual_counts(X, y, eta=0.5)
print("  sklearn(eta0=0.5) coef_     :", np.round(sk_half.coef_[0], 4))
print("  0.5 × Σ c_i·y_i·x_i         :", np.round(0.5 * (X * (c_half * y)[:, None]).sum(axis=0), 4))
print("  allclose                    :",
      bool(np.allclose(sk_half.coef_[0], 0.5 * (X * (c_half * y)[:, None]).sum(axis=0))))
print()

print("== 开 L2：同样的 c_i 再也重建不出 w ==")
w_reg, b_reg, c_reg = manual_counts(X, y, alpha=0.01)
gap = w_reg - (X * (c_reg * y)[:, None]).sum(axis=0)
print("  实际 w                      :", np.round(w_reg, 4))
print("  Σ c_i·y_i·x_i               :", np.round((X * (c_reg * y)[:, None]).sum(axis=0), 4))
print("  偏差范数                    :", float(np.linalg.norm(gap)))
print("  -> 收缩让 w 不再是样本的整数倍线性组合，α 表示的语义失效")


== 规模对照：原始 vs 对偶 ==
  n=60, d=2
  原始：每次判定 2 次乘加，状态 2 个数
  对偶：每次判定 60 次乘加，状态 60 个数，Gram 矩阵 60×60

== 证据：签名与属性里没有对偶 ==
  签名有 kernel / dual ? False / False
  fit 后属性         : ['classes_', 'coef_', 'intercept_', 'loss_function_', 'n_features_in_', 'n_iter_', 't_']
  有 dual_coef_ / support_ ? False / False
  SVC(kernel=linear)  : dual_coef_(1, 11), n_support_[5 6]

== 恒等式重建：eta=1、无正则 ==
  sklearn coef_               : [  5.0679 -12.8513]
  Σ c_i·y_i·x_i               : [  5.0679 -12.8513]
  最大偏差                    : 6.039613253960852e-14
  allclose                    : True
  intercept_                  : 33.0 | 由 c_i 重建: 33.0
  被用到的样本数 (c_i>0)      : 24 / 60
  -> 对偶系数是整数计数，但 sklearn 只存 w

== eta≠1：系数变成 eta × c_i ==


  sklearn(eta0=0.5) coef_     : [ 2.534  -6.4257]
  0.5 × Σ c_i·y_i·x_i         : [ 2.534  -6.4257]
  allclose                    : True

== 开 L2：同样的 c_i 再也重建不出 w ==


  实际 w                      :

 [ 2.4959 -8.6246]
  Σ c_i·y_i·x_i               : [ 333.6407 -825.4106]
  偏差范数                    : 881.3604200212343
  -> 收缩让 w 不再是样本的整数倍线性组合，α 表示的语义失效


## 7. 循环可以逐元素复现（有条件）

为便于与手工实现逐步对照，本实验使用**固定样本顺序、稠密输入、无正则化，并关闭提前停止**：

- `shuffle=False`（不洗牌）
- `tol=None`（不按 loss 早停，跑满 `max_iter`）
- 稠密输入（避开截距衰减）
- `penalty=None`（无正则）
- `early_stopping=False`

此外，两种实现采用相同的零初始化、单位学习率和样本权重。此时更新规则就是 `if y*(x@w+b) <= 0: w += y*x; b += y`。注意是 **`<=`**：带符号间隔恰好为 0 时也会更新。

为了防止模型找到解提前停止，实验数据需要**线性不可分**。`make_blobs` 的两簇只是有重叠，而重叠并不能排除存在一条线性分界，因此单靠它不能保证不可分。这里在数据末尾**加入**相同特征、相反标签的样本对。


In [10]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.linear_model import Perceptron


def manual_perceptron(X, y, max_epochs=1000, op="<="):
    """标准循环；op 用来对比 '<' 与 '<=' 两种判据。"""
    w = np.zeros(X.shape[1])
    b = 0.0
    updates = 0
    for epoch in range(1, max_epochs + 1):
        mistakes = 0
        for xi, yi in zip(X, y):
            z = yi * (xi @ w + b)
            hit = z < 0 if op == "<" else z <= 0
            if hit:
                w += yi * xi
                b += yi
                mistakes += 1
                updates += 1
        if mistakes == 0:
            return w, b, epoch, updates
    return w, b, max_epochs, updates


# 重叠区不能保证不可分；加入两组「同特征、相反标签」的矛盾样本对，
# 任何线性分界都不可能把它们同时分对 -> 不可分是确定的。
X_blob, y_blob = make_blobs(n_samples=200, centers=2, random_state=7, cluster_std=2.0)
y_blob = np.where(y_blob == 0, -1, 1)
X_pair = np.array([[0.0, 0.0], [0.0, 0.0], [5.0, 5.0], [5.0, 5.0]])
y_pair = np.array([-1, 1, -1, 1])
X = np.vstack([X_blob, X_pair])
y = np.concatenate([y_blob, y_pair])

w_m, b_m, epochs_m, updates_m = manual_perceptron(X, y)
sk = Perceptron(tol=None, shuffle=False, random_state=0, max_iter=1000).fit(X, y)

print("== 严格不可分数据（200 个 blobs 样本 + 2 组矛盾对）：手工循环 vs sklearn ==")
print("  手工   : epochs=%d, updates=%d, coef[:2]=%s, b=%.4f"
      % (epochs_m, updates_m, np.round(w_m[:2], 4), b_m))
print("  sklearn: n_iter_=%d, t_=%d, coef[:2]=%s, b=%.4f"
      % (sk.n_iter_, int(sk.t_), np.round(sk.coef_[0][:2], 4), sk.intercept_[0]))
print("  逐元素近似一致 (np.allclose) :",
      bool(np.allclose(w_m, sk.coef_[0]) and np.allclose(b_m, sk.intercept_[0])))
print("  实际更新次数 vs t_ :", updates_m, "vs", int(sk.t_))
print()

print("== 判据的等号：'<' 与 '<=' 谁对？ ==")
for op in ("<", "<="):
    w, b, ep, upd = manual_perceptron(X, y, op=op)
    same = bool(np.allclose(w, sk.coef_[0]) and np.allclose(b, sk.intercept_[0]))
    print("  手工用 '%s' : epochs=%d, updates=%d, 与 sklearn 近似一致=%s"
          % (op, ep, upd, same))
print("  用 '<' 时零初始化下第一个样本就不命中，循环因没有触发更新而提前结束")
print("  （提前结束不代表分类正确）")
print()

print("== 单样本边界：决策值恰好为 0 时是否更新 ==")
X0 = np.array([[0.0], [0.0]])
y0 = np.array([1, 0])
sw0 = np.array([1.0, 0.0])
p0 = Perceptron(shuffle=False, max_iter=1, tol=None).fit(
    X0, y0, sample_weight=sw0,
    coef_init=np.array([0.0]), intercept_init=np.array([0.0]))
print("  perceptron intercept_ =", p0.intercept_, "（从 0 变成 1 -> 0 分界也更新）")
print()

Xs, ys = make_blobs(n_samples=60, centers=2, random_state=1, cluster_std=0.4)
print("== 可分数据：tol 决定跑几轮 ==")
print("  默认 tol=1e-3 : n_iter_ =", Perceptron(shuffle=False, random_state=0).fit(Xs, ys).n_iter_)
print("  tol=None      : n_iter_ =", Perceptron(shuffle=False, random_state=0, tol=None).fit(Xs, ys).n_iter_)


== 严格不可分数据（200 个 blobs 样本 + 2 组矛盾对）：手工循环 vs sklearn ==
  手工   : epochs=1000, updates=18583, coef[:2]=[ 8.3589 -7.1631], b=75.0000
  sklearn: n_iter_=1000, t_=204001, coef[:2]=[ 8.3589 -7.1631], b=75.0000
  逐元素近似一致 (np.allclose) : True
  实际更新次数 vs t_ : 18583 vs 204001

== 判据的等号：'<' 与 '<=' 谁对？ ==
  手工用 '<' : epochs=1, updates=0, 与 sklearn 近似一致=False


  手工用 '<=' : epochs=1000, updates=18583, 与 sklearn 近似一致=True
  用 '<' 时零初始化下第一个样本就不命中，循环因没有触发更新而提前结束
  （提前结束不代表分类正确）

== 单样本边界：决策值恰好为 0 时是否更新 ==
  perceptron intercept_ = [1.] （从 0 变成 1 -> 0 分界也更新）

== 可分数据：tol 决定跑几轮 ==
  默认 tol=1e-3 : n_iter_ = 7
  tol=None      : n_iter_ = 1000


## 8. 文档里的「更新次数」应该怎么理解

`Perceptron` 的属性文档写：

> *Number of weight updates performed during training. Same as `(n_iter_ * n_samples + 1)`.*

这句话不能按字面读。`t_` 是**按迭代轮数和输入样本数维护的计数属性**，它不等同于参数实际改变的次数：

- **前一节的实验**里，204 个样本跑满 1000 轮 → `t_ = 1000 × 204 + 1`，但真正发生权重更新的只有 **18583** 次；
- 启用验证集（`early_stopping=True`）时，外层公式仍然使用**完整的输入样本数**，被切出去当验证集的那部分样本照样计入；
- 多分类时，`t_` 根据各二分类器的**最大**训练轮数计算，因此不能表示所有二分类器的累计样本处理次数。

所以稳妥的表述是：**`t_` 是按迭代轮数和输入样本数维护的计数属性，不能直接解释为参数实际改变的次数。**

注：`fit` 每次都把 `t_` 重置为 1.0；`partial_fit` 不重置、跨调用累加（下面的代码同时验证这两点）。


In [11]:
import warnings

import numpy as np
from sklearn.datasets import load_digits, make_blobs
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Perceptron

X, y = make_blobs(n_samples=60, centers=2, random_state=0)
p = Perceptron(random_state=0).fit(X, y)
print("== 二分类 ==")
print("  n_iter_=%d, n_samples=%d, t_=%d, 公式值=%d"
      % (p.n_iter_, len(y), int(p.t_), p.n_iter_ * len(y) + 1))

early = Perceptron(early_stopping=True, random_state=0, max_iter=1000).fit(X, y)
print("  early_stopping: n_iter_=%d, t_=%d, 公式值=%d"
      % (early.n_iter_, int(early.t_), early.n_iter_ * len(y) + 1))

Xd, yd = load_digits(return_X_y=True)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    m = Perceptron(random_state=0, max_iter=50).fit(Xd, yd)
print("== 多分类（digits）==")
print("  n_iter_=%d, n_samples=%d, t_=%d, 公式值=%d"
      % (m.n_iter_, len(yd), int(m.t_), m.n_iter_ * len(yd) + 1))

print()
print("== partial_fit：t_ 累积，n_iter_ 恒为 1 ==")
pf = Perceptron(random_state=0)
pf.partial_fit(X, y, classes=np.array([0, 1]))
print("  第 1 次：n_iter_=%d, t_=%d" % (pf.n_iter_, int(pf.t_)))
pf.partial_fit(X, y)
print("  第 2 次：n_iter_=%d, t_=%d" % (pf.n_iter_, int(pf.t_)))

print()
print("== warm_start：coef 复用，但 t_ 仍被重置 ==")
ws = Perceptron(warm_start=True, random_state=0, max_iter=2, tol=None).fit(X, y)
t1 = int(ws.t_)
ws.fit(X, y)
print("  第 1 次 fit t_=%d, 第 2 次 fit t_=%d" % (t1, int(ws.t_)))

== 二分类 ==
  n_iter_=15, n_samples=60, t_=901, 公式值=901
  early_stopping: n_iter_=7, t_=421, 公式值=421
== 多分类（digits）==
  n_iter_=22, n_samples=1797, t_=39535, 公式值=39535

== partial_fit：t_ 累积，n_iter_ 恒为 1 ==
  第 1 次：n_iter_=1, t_=61
  第 2 次：n_iter_=1, t_=121

== warm_start：coef 复用，但 t_ 仍被重置 ==
  第 1 次 fit t_=121, 第 2 次 fit t_=121


## 9. 多分类 = OvA + joblib：`n_jobs` 只在多分类生效

`Perceptron` 没有 softmax、也没有多类联合更新。`_partial_fit` 在 `n_classes > 2` 时走 `_fit_multiclass`：

- 每个类各训一个「本类 vs 其余」的二分类器；
- 先在外层用**原始多分类标签**算好验证掩码，所有 OvA job 共享；
- `Parallel(n_jobs=..., require="sharedmem")` 并行，每个 job 的随机种子在并行前统一抽取；
- 最后 `n_iter_` 取各二分类的**最大值**，不是总和。

因此 `n_jobs` 在二分类路径上完全没用——`_fit_binary` 的源码里根本不读它。


In [12]:
import inspect
import warnings

import numpy as np
from sklearn.datasets import load_digits
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Perceptron
from sklearn.linear_model._stochastic_gradient import BaseSGDClassifier

Xd, yd = load_digits(return_X_y=True)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    m = Perceptron(random_state=0, max_iter=50).fit(Xd, yd)

print("== digits 多分类 ==")
print("  coef_.shape      :", m.coef_.shape)
print("  intercept_.shape :", m.intercept_.shape)
print("  n_iter_          :", m.n_iter_, "（各二分类的最大值）")
print("  t_               :", int(m.t_), "=", m.n_iter_ * len(yd), "+ 1")
print()

print("== n_jobs 的作用范围 ==")
print("  '_fit_binary' 源码里出现 n_jobs ?", "n_jobs" in inspect.getsource(BaseSGDClassifier._fit_binary))
m1 = Perceptron(random_state=0, n_jobs=1, max_iter=10, tol=None).fit(Xd, yd)
m2 = Perceptron(random_state=0, n_jobs=-1, max_iter=10, tol=None).fit(Xd, yd)
print("  n_jobs=1 与 -1 的 coef_ 完全一致:", np.array_equal(m1.coef_, m2.coef_))
print("  两者 n_iter_:", m1.n_iter_, m2.n_iter_)

== digits 多分类 ==
  coef_.shape      : (10, 64)
  intercept_.shape : (10,)
  n_iter_          : 22 （各二分类的最大值）
  t_               : 39535 = 39534 + 1

== n_jobs 的作用范围 ==
  '_fit_binary' 源码里出现 n_jobs ? False
  n_jobs=1 与 -1 的 coef_ 完全一致: True
  两者 n_iter_: 10 10


## 10. 同一套「耐心」机制，只是监控指标不同

**`tol`**：停止条件不是「和上一轮比」，而是**和历史最优损失比**。源码 docstring 原文：

> *training will stop when `(loss > best_loss - tol)` for `n_iter_no_change` consecutive epochs.*

也就是说：**连续 `n_iter_no_change` 轮（默认 5）都没能把历史最优 loss 压低超过 `tol`，才停**，而不是「单轮下降不足 `tol` 就停」。它监控的是**训练** loss。`tol=None` 时阈值被换成 `-inf`，上式永不成立，等于永不早停；因为 `self.tol is not None` 不成立，连 `ConvergenceWarning` 都不会报。

**`early_stopping`**：用 `StratifiedShuffleSplit` 切出验证集，监控量从训练 loss 换成**验证集 score**，判据变成「验证 score 连续 `n_iter_no_change` 轮提升不足 `tol`」。所以两者是**同一套耐心计数机制，选择不同的监控指标**（越高越好 vs 越低越好）；`partial_fit` 直接禁止早停。

下面的实验：既看 `tol=None/1e-3` 的警告差异，也直接扫 `n_iter_no_change`，观察等待轮数如何影响停止时间。


In [13]:
import warnings

import numpy as np
from sklearn.datasets import make_blobs
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Perceptron

X, y = make_blobs(n_samples=120, centers=2, random_state=0, cluster_std=2.5)

print("== tol 不同取值下的警告 ==")
for tol in (None, 1e-3):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        est = Perceptron(max_iter=3, tol=tol, random_state=0).fit(X, y)
    names = [w.category.__name__ for w in caught]
    print("  tol=%-6r -> n_iter_=%d, 警告=%s" % (tol, est.n_iter_, names or "无"))

print()
print("== 耐心计数：n_iter_no_change 越大，停得越晚 ==")
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    for n in (1, 5, 20, 100):
        e = Perceptron(tol=1e-3, n_iter_no_change=n, max_iter=500, random_state=0).fit(X, y)
        print("  监控训练 loss : n_iter_no_change=%-4d -> n_iter_ = %d" % (n, e.n_iter_))
    for n in (1, 5, 20):
        e = Perceptron(early_stopping=True, tol=1e-3, n_iter_no_change=n,
                       max_iter=500, random_state=0).fit(X, y)
        print("  监控验证 score: n_iter_no_change=%-4d -> n_iter_ = %d" % (n, e.n_iter_))

print()
early = Perceptron(early_stopping=True, random_state=0, max_iter=1000).fit(X, y)
print("== early_stopping ==")
print("  n_iter_=%d, t_=%d, n_iter_*n+1=%d（外层仍用完整样本数 %d）"
      % (early.n_iter_, int(early.t_), early.n_iter_ * len(y) + 1, len(y)))

try:
    Perceptron(early_stopping=True).partial_fit(X, y, classes=np.array([0, 1]))
except Exception as e:
    print("  partial_fit + early_stopping ->", type(e).__name__, ":", str(e).splitlines()[0])


== tol 不同取值下的警告 ==
  tol=None   -> n_iter_=3, 警告=无
  tol=0.001  -> n_iter_=3, 警告=['ConvergenceWarning']

== 耐心计数：n_iter_no_change 越大，停得越晚 ==
  监控训练 loss : n_iter_no_change=1    -> n_iter_ = 2
  监控训练 loss : n_iter_no_change=5    -> n_iter_ = 11
  监控训练 loss : n_iter_no_change=20   -> n_iter_ = 45
  监控训练 loss : n_iter_no_change=100  -> n_iter_ = 234
  监控验证 score: n_iter_no_change=1    -> n_iter_ = 2
  监控验证 score: n_iter_no_change=5    -> n_iter_ = 11
  监控验证 score: n_iter_no_change=20   -> n_iter_ = 26

== early_stopping ==
  n_iter_=11, t_=1321, n_iter_*n+1=1321（外层仍用完整样本数 120）
  partial_fit + early_stopping -> ValueError : early_stopping should be False with partial_fit


## 11. 稀疏输入采用较小的截距更新步长

`make_dataset` 对稠密输入返回 `intercept_decay=1.0`，对稀疏输入返回 `SPARSE_INTERCEPT_DECAY=0.01`（源码注释：*avoid intercept oscillation*）。

在更新量相同的情况下，稀疏路径将单次截距增量乘以 0.01。随着训练继续，两种路径的模型参数和后续更新可能不同，因此最终截距不一定相差 100 倍。

结果：**同一份数据、同一组超参，`csr_matrix` 与稠密数组可能得到不同的模型**。感知机默认不加正则，截距更新的差异会影响后续样本的判断，从而改变系数的更新轨迹。


In [14]:
import numpy as np
import scipy.sparse as sp
from sklearn.datasets import make_blobs
from sklearn.linear_model import Perceptron
from sklearn.linear_model._base import SPARSE_INTERCEPT_DECAY, make_dataset

print("SPARSE_INTERCEPT_DECAY =", SPARSE_INTERCEPT_DECAY)

X, y = make_blobs(n_samples=100, centers=2, random_state=3, cluster_std=1.5)
yf = y.astype(np.float64)
sw = np.ones(len(y))
print("make_dataset 的 intercept_decay: dense=%s, sparse=%s"
      % (make_dataset(X, yf, sw)[1], make_dataset(sp.csr_matrix(X), yf, sw)[1]))

kw = dict(shuffle=False, random_state=0, max_iter=30, tol=None)
d = Perceptron(**kw).fit(X, y)
s = Perceptron(**kw).fit(sp.csr_matrix(X), y)

print()
print("%-8s %-24s %-12s %-8s" % ("输入", "coef_ 前两维", "intercept_", "t_"))
for name, est in (("dense", d), ("csr", s)):
    print("%-8s %-24s %-12.2f %-8d"
          % (name, np.round(est.coef_[0][:2], 3), est.intercept_[0], int(est.t_)))
print()
print("coef 是否一致:", np.allclose(d.coef_, s.coef_))

SPARSE_INTERCEPT_DECAY = 0.01
make_dataset 的 intercept_decay: dense=1.0, sparse=0.01

输入       coef_ 前两维                intercept_   t_      
dense    [-9.903 -7.036]          3.00         3001    
csr      [-9.376 -8.868]          0.14         3001    

coef 是否一致: False


## 12. 权重不等价于重采样；感知机没有概率

**`class_weight`**：`class_weight="balanced"` 只算出每类一个**实数**权重（本实验是 `[0.5952, 3.125]`，权重比为 1 : 5.25），数据不复制、行数不变；权重按 `sample_weight` **乘性**叠加到每次更新的幅度上（实测把所有样本权重乘 2，`coef_` 也翻倍，见下）。

它和「复制样本」**不等价**。比例本身是可以对齐的：多数类与少数类的复制倍数分别为 **4 和 21**，其比例与类别权重比例一致；复制后两类样本数均为 **1008**，类别数量比为 **1 : 1**。真正的区别在**在线学习**的机制上：

- `balanced` 按类别权重**缩放单次更新的幅度**（多数类权重 0.5952 < 1，少数类 3.125 > 1），一次更新算一步；
- 复制改变的是**访问次数与遍历顺序**（每个 epoch 的样本数不同、洗牌后的排列也不同），一次更新只算一步。

感知机每步都在改 `w`，后续样本的判定取决于当前 `w`。所以「缩放一次更新」和「重复访问样本」**通常不能保证相同的训练轨迹**；实测两个坐标的系数比值分别为 **0.171 和 0.213**，不是同一个常数。

**API**：`Perceptron` 没有 `predict_proba`，可以通过 `decision_function` 获得决策分数，但这些分数不能直接解释为概率。二分类返回 1D，多分类返回 2D，`predict` 走符号 / `argmax`。


In [15]:
import numpy as np
from sklearn.datasets import load_digits, make_blobs, make_classification
from sklearn.linear_model import Perceptron
from sklearn.utils.class_weight import compute_class_weight

X, y = make_classification(n_samples=300, n_features=2, n_informative=2, n_redundant=0,
                           weights=[0.85, 0.15], class_sep=1.2, flip_y=0.03, random_state=0)
cnt = np.bincount(y)
w = compute_class_weight("balanced", classes=np.array([0, 1]), y=y)
print("== 不平衡数据 ==")
print("  类别样本数   :", cnt)
print("  balanced 权重:", np.round(w, 4), "-> 权重比 1 :", round(w[1] / w[0], 3))

# 权重是"乘性"的：整体缩放所有样本权重，等价于缩放每一次更新的幅度
Xw, yw = make_blobs(n_samples=120, centers=2, random_state=0, cluster_std=2.5)
kwd = dict(shuffle=False, random_state=0, max_iter=1, tol=None)
w1 = Perceptron(**kwd).fit(Xw, yw).coef_[0]
w2 = Perceptron(**kwd).fit(Xw, yw, sample_weight=np.full(len(yw), 2.0)).coef_[0]
print()
print("== 权重是乘性的：全体 sample_weight 乘 2 ==")
print("  coef(sw=1) =", np.round(w1, 4))
print("  coef(sw=2) =", np.round(w2, 4), "  比值 =", np.round(w2 / w1, 4))

# 复制倍数与类别权重比例对齐：权重比 5.25 = 21/4 -> 多数类 ×4、少数类 ×21
maj, mino = int(np.argmax(cnt)), int(np.argmin(cnt))
k = {maj: 4, mino: 21}
X_rep = np.vstack([np.repeat(X[y == c], k[c], axis=0) for c in (0, 1)])
y_rep = np.concatenate([np.repeat(y[y == c], k[c]) for c in (0, 1)])
print()
print("== 按类别权重比例设定复制倍数：多数类(类 %d) ×%d，少数类(类 %d) ×%d =="
      % (maj, k[maj], mino, k[mino]))
print("  原始样本数 %s -> 复制后 %s（类别数量比 %s）"
      % (cnt, np.bincount(y_rep),
         np.round(np.bincount(y_rep).astype(float) / np.bincount(y_rep).min(), 3)))

fit_kw = dict(random_state=0, max_iter=200, tol=None)
bal = Perceptron(class_weight="balanced", **fit_kw).fit(X, y)
rep = Perceptron(**fit_kw).fit(X_rep, y_rep)
print()
print("  class_weight='balanced' : coef=%s, train score=%.3f"
      % (np.round(bal.coef_[0], 3), bal.score(X, y)))
print("  复制多数×%d / 少数×%d    : coef=%s, train score=%.3f"
      % (k[maj], k[mino], np.round(rep.coef_[0], 3), rep.score(X, y)))
print()
print("  coef 近似相同 (np.allclose) :", np.allclose(bal.coef_, rep.coef_))
print("  coef 两坐标比值 rep/bal     :", np.round(rep.coef_[0] / bal.coef_[0], 3),
      "-> 不是同一个常数，不能视为等比缩放")
print("  -> balanced 按类别权重缩放单次更新；复制改变访问次数与顺序")
print("  -> 两者通常不能保证相同的训练轨迹")

try:
    Perceptron(class_weight="balanced").partial_fit(X, y, classes=np.array([0, 1]))
except Exception as e:
    print()
    print("  partial_fit + balanced ->", type(e).__name__, ":", str(e).splitlines()[0][:80])

Xb, yb = make_blobs(n_samples=40, centers=2, random_state=0)
pb = Perceptron(random_state=0).fit(Xb, yb)
print()
print("== API 边界 ==")
print("  有 predict_proba 吗？        :", hasattr(pb, "predict_proba"))
print("  二分类 decision_function.shape:", pb.decision_function(Xb).shape)
print("  二分类 predict.shape          :", pb.predict(Xb).shape)

Xd, yd = load_digits(return_X_y=True)
mdl = Perceptron(random_state=0, max_iter=5, tol=None).fit(Xd, yd)
print("  多分类 decision_function.shape:", mdl.decision_function(Xd).shape)


== 不平衡数据 ==
  类别样本数   : [252  48]
  balanced 权重: [0.5952 3.125 ] -> 权重比 1 : 5.25

== 权重是乘性的：全体 sample_weight 乘 2 ==
  coef(sw=1) = [-2.562  -1.0226]
  coef(sw=2) = [-5.124  -2.0453]   比值 = [2. 2.]



== 按类别权重比例设定复制倍数：多数类(类 0) ×4，少数类(类 1) ×21 ==
  原始样本数 [252  48] -> 复制后 [1008 1008]（类别数量比 [1. 1.]）

  class_weight='balanced' : coef=[ 8.829 -5.29 ], train score=0.720
  复制多数×4 / 少数×21    : coef=[ 1.506 -1.128], train score=0.813

  coef 近似相同 (np.allclose) : False
  coef 两坐标比值 rep/bal     : [0.171 0.213] -> 不是同一个常数，不能视为等比缩放
  -> balanced 按类别权重缩放单次更新；复制改变访问次数与顺序
  -> 两者通常不能保证相同的训练轨迹

  partial_fit + balanced -> ValueError : class_weight 'balanced' is not supported for partial_fit. In order to use 'balan

== API 边界 ==
  有 predict_proba 吗？        : False
  二分类 decision_function.shape: (40,)
  二分类 predict.shape          : (40,)
  多分类 decision_function.shape: (1797, 10)


## 13. 小结

| # | 现象 | 分析 |
|---|---|---|
| 1 | 文件 221 行 / 类体 213 行（docstring 158 行） | 公开参数缩减：签名里没有 `loss`、`average`；`loss`/`learning_rate` 固定，`eta0` 默认 1.0 可改 |
| 2 | `perceptron → Hinge(0.0)`，`hinge → Hinge(1.0)` | 两者只差一个阈值，且折点处取等号（`z <= θ` 就更新） |
| 3 | `shuffle=False + tol=None` 时手工循环近似一致（`np.allclose`） | 边界是 `z <= 0`；不可分性靠「相反标签样本对」严格保证 |
| 4 | `t_` 按「迭代轮数 × 输入样本数」维护 | 文档说「更新次数」，但不能等同于参数实际改变次数 |
| 5 | 多分类 OvA + joblib | `n_jobs` 只在多分类生效，`n_iter_` 取最大值 |
| 6 | `tol` 与**历史最优** loss 比较 | 连续 `n_iter_no_change` 轮改善不足才停；`tol=None` 静默跑满 |
| 7 | `early_stopping` 看验证集 | 同一套耐心计数，只是换成监控验证 score |
| 8 | 稀疏输入截距步长衰减 0.01 | 同一数据 dense 与 csr 可能得到不同模型（`b` 3.0 vs 0.14） |
| 9 | `class_weight` 是乘性权重 | 按类别权重缩放单次更新 ≠ 重复访问样本，与重采样不等价 |
| 10 | 没有 `predict_proba` | `decision_function` 给出决策分数，但不能直接解释为概率 |
| 11 | 没有对偶形式 | 只维护 `w`；`w = Σ c_i y_i x_i` 可事后重建，开正则或改步长后失效 |

`Perceptron` 的定位是：复用基类训练实现，固定 `loss` 与 `learning_rate`，缩减 `loss` / `average` 两个公开参数，并给 `eta0`、`penalty`、`shuffle`、`tol`、`early_stopping` 等提供默认值。因此默认配置对应原始的标准感知机，但其中若干开关仍可由用户调整。
